In [ ]:
filename = 'companyPolicy.txt'

file= open(filename)
Loader = TextLoader(file)
documents = Loader.load()
text_splitter = CharacterTextSplitter(chunk_size = 100, chunk_overlap = 0, separator = "\n")
texts = text_splitter.split_documnets(documents)
embeddings = HuggingFaceEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings)
print('document embedded')
params = {
    GenTextParamsMetaNames.TEMPERATURE = 0.5,
    GenTextParamsMetaNames.MAX_NEW_TOKENS = 512
}
project_id = "skills-network"
url = "https://us-south-ibm.ml.cloud.com"
model_id = "llamavision-3b-instruct"
model = WatsonXLLM(model_id= model_id, url = url, params = params, project_id = project_id)


In [ ]:
prompt_template = """
Use information to answer the question. When not found say I dont know
Question: {question}
Context: {context}
"""

prompt = PromptTemplate(template = prompt_template, input_variables= ["context", "question"])


In [ ]:
memory = ConversationBufferMemory(memory_key= "chat_history", return_message= True)


In [ ]:
qa = ConversationalRetrievalChain.from_llm(llm=llm, chain_type= "stuff", 
                                           retriever= docsearch.as_retriever(), 
                                           memory= memory, get_chat_history= lambda h : h, 
                                           return_source_documents= False)
history = []
query = "How can i ask for leaves"
response = qa.invoke({"question" : query, {"chat_history": history}})
print(response["answer"])

In [ ]:
history.append((query, result("answer")))
query = "What are the key points in it. Summarize"
response = qa.invoke({"question" : query}, {"chat_history": history})
history.append(query, response["answer"])

In [ ]:
file.close()